In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [53]:
!pip install mlflow dagshub -q

import mlflow
import mlflow.sklearn
import dagshub
import warnings
from kaggle_secrets import UserSecretsClient

user_secrets     = UserSecretsClient()
dagshub_token    = user_secrets.get_secret("ML_assn1")
dagshub_username = user_secrets.get_secret("username")

os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

repo_owner = dagshub_username
repo_name  = "ml_assn1"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

RUN_ID       = "db18ca91aceb46e984b8c4ef4b5ac0cb"
MODEL_NAME   = "house-prices-gb"
MODEL_VERSION = 4
VAL_RMSE     = 0.13589     # logged during training

COMPETITION  = "house-prices-advanced-regression-techniques"

print("Connected to DagsHub MLflow")
print(f"Model  : {MODEL_NAME}  v{MODEL_VERSION}  (run: {RUN_ID[:8]}...)")

Initialized MLflow to track repo "lkuch23/ml_assn1"

Repository lkuch23/ml_assn1 initialized!

Connected to DagsHub MLflow
Model  : house-prices-gb  v4  (run: db18ca91...)


In [54]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")
    model = mlflow.sklearn.load_model(model_uri)

In [55]:
_paths = [
    "/kaggle/input/house-prices-advanced-regression-techniques",
    "/kaggle/input/competitions/house-prices-advanced-regression-techniques",
]
DATA_DIR = next((p for p in _paths if os.path.exists(f"{p}/train.csv")), None)
if DATA_DIR is None:
    os.makedirs("/kaggle/working/data", exist_ok=True)
    os.system(f"kaggle competitions download -c {COMPETITION} -p /kaggle/working/data --unzip")
    DATA_DIR = "/kaggle/working/data"

train_raw = pd.read_csv(f"{DATA_DIR}/train.csv")
test_raw  = pd.read_csv(f"{DATA_DIR}/test.csv")
test_ids  = test_raw["Id"].copy()

train_num_medians = (
    train_raw.drop("SalePrice", axis=1)
    .select_dtypes(include=["int64", "float64"])
    .median()
)

train = train_raw.drop("SalePrice", axis=1).copy()
num_cols = train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = train.select_dtypes(include=["object"]).columns
train[num_cols] = train[num_cols].fillna(train[num_cols].median())
train[cat_cols] = train[cat_cols].fillna("None")
train["TotalSF"]     = train["TotalBsmtSF"] + train["1stFlrSF"] + train["2ndFlrSF"]
train["TotalBathSF"] = train["FullBath"] + train["BsmtFullBath"] + 0.5 * (train["HalfBath"] + train["BsmtHalfBath"])
train["TotalRooms"]  = train["TotRmsAbvGrd"] + train["FullBath"] + train["HalfBath"]
train = pd.get_dummies(train)

test = test_raw.copy()
for col in test.select_dtypes(include=["int64", "float64"]).columns:
    fill = train_num_medians[col] if col in train_num_medians.index else test[col].median()
    test[col] = test[col].fillna(fill)
test[test.select_dtypes(include=["object"]).columns] = test.select_dtypes(include=["object"]).fillna("None")
test["TotalSF"]     = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]
test["TotalBathSF"] = test["FullBath"] + test["BsmtFullBath"] + 0.5 * (test["HalfBath"] + test["BsmtHalfBath"])
test["TotalRooms"]  = test["TotRmsAbvGrd"] + test["FullBath"] + test["HalfBath"]
test = pd.get_dummies(test)

X_test = test.reindex(columns=train.columns, fill_value=0)

In [56]:
log_preds   = model.predict(X_test)
sale_prices = np.expm1(log_preds)

In [57]:
submission = pd.DataFrame({"Id": test_ids.values, "SalePrice": sale_prices})

SUBMISSION_PATH = "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

In [58]:
!kaggle competitions submit \
    -c {COMPETITION} \
    -f {SUBMISSION_PATH} \
    -m "{MESSAGE}"

100%|███████████████████████████████████████| 33.6k/33.6k [00:00<00:00, 191kB/s]
Successfully submitted to House Prices - Advanced Regression Techniques

In [59]:
import time, subprocess
from io import StringIO

time.sleep(30)

result = subprocess.run(
    ["kaggle", "competitions", "submissions", "-c", COMPETITION, "--csv"],
    capture_output=True, text=True,
)

clean_csv = "\n".join(
    line for line in result.stdout.splitlines()
    if not line.startswith("Warning:")
)

subs_df = pd.read_csv(StringIO(clean_csv))
latest  = subs_df.iloc[0]

public_score = float(latest["publicScore"])
status       = str(latest["status"])

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("kaggle_public_rmse", public_score)

🏃 View run rebellious-cub-436 at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0/runs/db18ca91aceb46e984b8c4ef4b5ac0cb
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0
